In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import glob

In [3]:
root = r"C:\Users\oxcy\Desktop\EMTEQ PROEKT\prviot code vtor pat\FEIT_eating_S05"
subjects = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]

In [4]:
subjects

['Andrej Petrov',
 'Angela Nastovska',
 'Bojan Dimovski',
 'Bojan Radovski',
 'Damjan Srebrenkoski',
 'Daniela Kovachovska',
 'Filip Sivevski',
 'Gorica Kovachovska',
 'Ilina Kovachovska',
 'Iva Jovanova',
 'Ivana Kiprijanovska',
 'Jovana Kostadinovska',
 'Kristijan Milosheski',
 'Marija Kovachovska',
 'Marko Kostov',
 'Matej Zlatkov',
 'Monika Stoilkovska',
 'Nikola Dimovski',
 'Ognen Sekuloski',
 'Sandra Shandarovska',
 'Sara Ilievska',
 'Sara Kovachovska',
 'Sashko Kovachovski',
 'Stefan Dinushev',
 'Stefanija Lazarovska',
 'Tarek Abd El-Azis',
 'Teodora Domazetovikj',
 'Tomi Jovanov',
 'Tomi Nikoloski',
 'Vasko Dimitrovski',
 'Vedrana Petreska',
 'Vladimir Petrov']

In [5]:
random.seed(42)
random.shuffle(subjects)

train_subjects = subjects[:22]
val_subjects   = subjects[22:27]
test_subjects  = subjects[27:32]

print("TRAIN:", len(train_subjects), train_subjects)
print("VAL:",   len(val_subjects), val_subjects)
print("TEST:",  len(test_subjects), test_subjects)

TRAIN: 22 ['Teodora Domazetovikj', 'Daniela Kovachovska', 'Ivana Kiprijanovska', 'Matej Zlatkov', 'Tarek Abd El-Azis', 'Jovana Kostadinovska', 'Sashko Kovachovski', 'Filip Sivevski', 'Sandra Shandarovska', 'Kristijan Milosheski', 'Monika Stoilkovska', 'Iva Jovanova', 'Tomi Nikoloski', 'Marko Kostov', 'Stefanija Lazarovska', 'Sara Ilievska', 'Vedrana Petreska', 'Angela Nastovska', 'Marija Kovachovska', 'Ognen Sekuloski', 'Bojan Dimovski', 'Nikola Dimovski']
VAL: 5 ['Sara Kovachovska', 'Bojan Radovski', 'Vasko Dimitrovski', 'Damjan Srebrenkoski', 'Tomi Jovanov']
TEST: 5 ['Vladimir Petrov', 'Ilina Kovachovska', 'Stefan Dinushev', 'Andrej Petrov', 'Gorica Kovachovska']


In [6]:
subject_csv = {}

for s in subjects:
    subj_dir = os.path.join(root, s)
    csvs = glob.glob(os.path.join(subj_dir, "eating_processed.csv"))

    if len(csvs) == 0:
        print(f" NO eating_processed for {s}")
    elif len(csvs) > 1:
        print(f" MORE than one for {s}, using first")
        subject_csv[s] = csvs[0]
    else:
        subject_csv[s] = csvs[0]

print("Total subjects with eating_processed:", len(subject_csv))

Total subjects with eating_processed: 32


### Funkcija za vcituvanje DF

In [7]:
def load_subject_df(subject):
    csv_path = subject_csv[subject]
    df = pd.read_csv(csv_path)
    df["subject"] = subject
    return df

In [8]:
train_dfs = [load_subject_df(s) for s in train_subjects if s in subject_csv]
val_dfs   = [load_subject_df(s) for s in val_subjects   if s in subject_csv]
test_dfs  = [load_subject_df(s) for s in test_subjects  if s in subject_csv]

print(len(train_dfs), len(val_dfs), len(test_dfs))

22 5 5


In [9]:
cols_to_drop = [
    "SoftwareTimestamp",
    "Timestamp",
    "Accelerometer/Raw.Z",
    "Gyroscope/Raw.Z",
    "Magnetometer/Raw.X",
    "Magnetometer/Raw.Y",
    "Magnetometer/Raw.Z",
    "Nav/Shutter[LeftTemple]",
    "Nav/FrameAvg[LeftTemple]",
    "Pressure/Raw", 
    "RotationVector/Raw.W",
    "RotationVector/Raw.Z",
]

train_dfs = [df.drop(columns=cols_to_drop, errors="ignore") for df in train_dfs]
val_dfs   = [df.drop(columns=cols_to_drop, errors="ignore") for df in val_dfs]
test_dfs  = [df.drop(columns=cols_to_drop, errors="ignore") for df in test_dfs]

In [10]:
train_df = pd.concat(train_dfs, ignore_index=True)
val_df   = pd.concat(val_dfs,   ignore_index=True)
test_df  = pd.concat(test_dfs,  ignore_index=True)

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["subject"].nunique(),
      val_df["subject"].nunique(),
      test_df["subject"].nunique())

(705651, 12) (153829, 12) (136385, 12)
22 5 5


In [ ]:
import numpy as np

def make_windows_from_dfs(
    dfs,
    label_col,
    feature_cols=None,
    window_samples=150,
    step_samples=25,
    label_strategy="majority"
):
    X_list, y_list = [], []

    for df in dfs:
        
        if feature_cols is None:
            num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            feature_cols = [c for c in num_cols if c != label_col]

        X_raw = df[feature_cols].to_numpy(dtype=np.float32)
        y_raw = df[label_col].to_numpy()

        n = len(X_raw)
        if n < window_samples:
            continue

        for start in range(0, n - window_samples + 1, step_samples):
            end = start + window_samples
            Xw = X_raw[start:end]

            y_seg = y_raw[start:end]
            if label_strategy == "last":
                yw = y_seg[-1]
            else:
                vals, counts = np.unique(y_seg, return_counts=True)
                yw = vals[np.argmax(counts)]

            X_list.append(Xw)
            y_list.append(yw)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list)
    return X, y

In [ ]:
label_col   = "Annotations"
subject_col = "subject"
time_col    = "SoftwareTimestamp"  

exclude = {label_col, subject_col, "SoftwareTimestamp", "Timestamp"}
feature_cols = [c for c in train_df.columns if c not in exclude]

print("n_features =", len(feature_cols))
print("first features:", feature_cols)

n_features = 10
first features: ['Accelerometer/Raw.X', 'Accelerometer/Raw.Y', 'Gyroscope/Raw.X', 'Gyroscope/Raw.Y', 'RotationVector/Raw.X', 'RotationVector/Raw.Y', 'Nav/Raw.X[LeftTemple]', 'Nav/Raw.Y[LeftTemple]', 'Prox/Raw[LeftTemple]', 'Nav/IQ[LeftTemple]']


In [13]:
LABEL_COL = "Annotations"     
WINDOW_SAMPLES = 200      # ~4s @ 50Hz
STEP_SAMPLES = 25         # ~0.5s

In [14]:
X_train, y_train = make_windows_from_dfs(
    train_dfs,
    feature_cols=feature_cols, # bez subject i anotations
    label_col=LABEL_COL,
    window_samples=WINDOW_SAMPLES,
    step_samples=STEP_SAMPLES
)

X_val, y_val = make_windows_from_dfs(
    val_dfs,
    feature_cols=feature_cols,
    label_col=LABEL_COL,
    window_samples=WINDOW_SAMPLES,
    step_samples=STEP_SAMPLES
)

X_test, y_test = make_windows_from_dfs(
    test_dfs,
    feature_cols=feature_cols,
    label_col=LABEL_COL,
    window_samples=WINDOW_SAMPLES,
    step_samples=STEP_SAMPLES
)

print(X_train.shape, X_val.shape, X_test.shape) #windows

(28061, 200, 10) (6116, 200, 10) (5419, 200, 10)


In [15]:
print(train_dfs[0].columns)

Index(['Accelerometer/Raw.X', 'Accelerometer/Raw.Y', 'Gyroscope/Raw.X',
       'Gyroscope/Raw.Y', 'RotationVector/Raw.X', 'RotationVector/Raw.Y',
       'Nav/Raw.X[LeftTemple]', 'Nav/Raw.Y[LeftTemple]',
       'Prox/Raw[LeftTemple]', 'Nav/IQ[LeftTemple]', 'Annotations', 'subject'],
      dtype='object')


In [17]:
mean = X_train.mean(axis=(0,1), keepdims=True)
std  = X_train.std(axis=(0,1), keepdims=True)
std[std == 0] = 1

X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

print("train mean first 5 feats:", X_train.mean(axis=(0,1))[:5])
print("train std  first 5 feats:", X_train.std(axis=(0,1))[:5])

train mean first 5 feats: [ 4.6527488e-04 -2.2359291e-01  5.7935631e-07 -2.2583384e-07
 -5.6996085e-03]
train std  first 5 feats: [1.0002505  0.9735126  0.99983466 0.9992957  0.99972796]


In [18]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [19]:
from sklearn.metrics import confusion_matrix, classification_report

In [20]:
from sklearn.metrics import roc_curve

In [ ]:
# model 2
input_shape = (X_train.shape[1], X_train.shape[2])  # (200, 12)

model2 = models.Sequential([
    layers.Input(shape=input_shape),

    layers.Conv1D(64, 7, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.2),

    layers.Conv1D(128, 5, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.2),

    layers.Conv1D(256, 3, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ]
)

model2.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 200, 64)        │         4,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 200, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 200, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 50, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 50, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 162,497 (634.75 KB)

 Trainable params: 161,601 (631.25 KB)

 Non-trainable params: 896 (3.50 KB)

In [22]:
from sklearn.utils.class_weight import compute_class_weight

In [52]:
classes = np.unique(y_train)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
class_weight # dali klasite se ramnomerni

{0: 1.3180366369187413, 1: 0.8056097841065687}

In [53]:
# def binary_focal_loss(gamma=2.0, alpha=0.25):
#     def loss(y_true, y_pred):
#         y_true = tf.cast(y_true, tf.float32)
#         y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
#         p_t = y_true*y_pred + (1-y_true)*(1-y_pred)
#         alpha_t = y_true*alpha + (1-y_true)*(1-alpha)
#         return -alpha_t * tf.pow(1.0 - p_t, gamma) * tf.math.log(p_t)
#     return loss
# # dava podobri rez so ovoj cell
# model2.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-3),
#     loss=binary_focal_loss(gamma=2.0, alpha=0.25),
#     metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()]
# )

In [23]:
cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                     patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max",
                                         factor=0.5, patience=3, min_lr=1e-6)
]

history2 = model2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=256,
    #class_weight=class_weight,  # ako sakash go dodavame tuka
    callbacks=cb,
    verbose=1
)

Epoch 1/60
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.8321 - auc: 0.9015 - loss: 0.3809 - precision: 0.8501 - recall: 0.8856 - val_accuracy: 0.8702 - val_auc: 0.9324 - val_loss: 0.3370 - val_precision: 0.9261 - val_recall: 0.8855 - learning_rate: 0.0010
Epoch 2/60
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.8720 - auc: 0.9381 - loss: 0.3035 - precision: 0.8794 - recall: 0.9200 - val_accuracy: 0.8697 - val_auc: 0.9355 - val_loss: 0.3120 - val_precision: 0.9083 - val_recall: 0.9056 - learning_rate: 0.0010
Epoch 3/60
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.8800 - auc: 0.9459 - loss: 0.2844 - precision: 0.8881 - recall: 0.9230 - val_accuracy: 0.8805 - val_auc: 0.9402 - val_loss: 0.2929 - val_precision: 0.9062 - val_recall: 0.9254 - learning_rate: 0.0010
Epoch 4/60
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.8841 - auc: 0.9507 - loss: 0.2713 - precision: 0.8893 - recall: 0.9289 - val_accuracy: 0.8797 - val_auc: 0.9323 - val_loss: 0.3398 -

In [26]:
y_prob = model2.predict(X_test).ravel()
y_pred = (y_prob >= 0.482).astype(int)

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=4))

170/170 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Confusion matrix:
 [[1141  254]
 [ 363 3661]]
              precision    recall  f1-score   support

           0     0.7586    0.8179    0.7872      1395
           1     0.9351    0.9098    0.9223      4024

    accuracy                         0.8861      5419
   macro avg     0.8469    0.8639    0.8547      5419
weighted avg     0.8897    0.8861    0.8875      5419



In [25]:
fpr, tpr, thr = roc_curve(y_test, y_prob)
best = thr[(tpr - fpr).argmax()]
print(best)

0.4815953


# Vaka go zachuvuvame modelot

In [27]:
model2.save("base_old_model.keras")